# 10. 샌드박스와 ACP

## 학습 목표
- 샌드박스 격리 개념과 보안 원칙을 이해한다
- E2B, Modal, Docker 등 샌드박스 프로바이더를 비교한다
- ACP(Agent Communication Protocol)의 개요와 용도를 안다
- 에이전트-에디터 통합 패턴을 이해한다
- 샌드박스 + ACP 통합 아키텍처를 설계한다

In [ ]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"모델 설정 완료: {model.model_name}")

---
## 1. 샌드박스 개념

**샌드박스**는 AI 에이전트가 코드를 실행하고, 파일을 관리하고, 쉘 명령을 수행할 수 있는 **격리된 실행 환경**입니다.

### 왜 격리가 중요한가?

| 위험 | 격리 없을 때 | 샌드박스 사용 시 |
|------|------------|----------------|
| 파일 시스템 접근 | 호스트 파일 변경/삭제 가능 | 격리된 파일시스템만 접근 |
| 네트워크 접근 | 무제한 외부 통신 | 제한된 네트워크 접근 |
| 자격 증명 | 환경 변수 유출 가능 | 시크릿 격리 |
| 시스템 영향 | 호스트 OS에 영향 | 호스트 시스템 보호 |

Deep Agents에서 샌드박스는 **백엔드**로 기능하며, 파일시스템 도구(`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`)와 `execute` 도구를 제공합니다.

---
## 2. 아키텍처 패턴

샌드박스 통합에는 두 가지 주요 패턴이 있습니다.

### Agent-in-Sandbox
에이전트가 샌드박스 **내부**에서 실행되며, 네트워크 프로토콜을 통해 외부와 통신합니다.

| 장점 | 단점 |
|------|------|
| 개발 환경과 동일한 경험 | 자격 증명 노출 위험 |
| 간단한 설정 | 인프라 복잡성 증가 |

### Sandbox-as-Tool (권장)
에이전트가 **외부**에서 실행되며, 샌드박스 API를 호출하여 코드를 실행합니다.

| 장점 | 단점 |
|------|------|
| 에이전트 상태와 실행 분리 | 네트워크 지연 |
| 시크릿을 샌드박스 외부에 유지 | |
| 병렬 태스크 실행 가능 | |

In [ ]:
# 두 가지 아키텍처 패턴 비교 (참고용)
print("=== 패턴 1: Agent-in-Sandbox ===")
print("  [샌드박스]")
print("    |-- 에이전트 (내부 실행)")
print("    |-- 파일시스템")
print("    |-- 코드 실행")
print("    <---> 네트워크 프로토콜 <---> 외부 시스템")

print()
print("=== 패턴 2: Sandbox-as-Tool (권장) ===")
print("  [호스트]")
print("    |-- 에이전트 (외부 실행)")
print("    |-- 자격 증명 관리")
print("    |-- API 호출 --> [샌드박스]")
print("                       |-- 파일시스템")
print("                       |-- 코드 실행")

---
## 3. 샌드박스 프로바이더 비교

| 프로바이더 | 특징 | 적합한 용도 |
|-----------|------|------------|
| **Modal** | GPU 지원, ML 워크로드 | AI/ML 작업, 데이터 처리 |
| **Daytona** | TypeScript/Python, 빠른 콜드 스타트 | 웹 개발, 빠른 반복 |
| **Runloop** | 일회용 devbox, 격리 실행 | 코드 테스트, 일회성 작업 |

In [ ]:
# Modal 샌드박스 설정 예시 (참고용)
modal_config = {
    "provider": "modal",
    "image": "python:3.12-slim",
    "gpu": "T4",
    "timeout": 300,
}

print("=== Modal 샌드박스 설정 ===")
for key, value in modal_config.items():
    print(f"  {key}: {value}")

print()
print("코드 예시 (참고용):")
print("  from deepagents.backends.sandbox import ModalSandbox")
print("  agent = create_deep_agent(")
print('      model="gpt-4.1",')
print('      backend=ModalSandbox(image="python:3.12-slim", gpu="T4"),')
print("  )")

---
## 4. 보안 가이드라인

### 절대 샌드박스 안에 시크릿을 넣지 마세요

컨텍스트에 주입된 에이전트는 환경 변수나 마운트된 파일로 저장된 자격 증명을 읽고 유출할 수 있습니다.

### 안전한 관행

1. **자격 증명은 외부 도구에서만 관리** — 샌드박스 외부의 전용 도구 사용
2. **Human-in-the-Loop** — 민감한 작업에 대해 사람 승인 요구
3. **네트워크 접근 차단** — 불필요한 아웃바운드 연결 차단
4. **아웃바운드 모니터링** — 예기치 않은 외부 연결 감시
5. **출력 검토** — 샌드박스 출력을 애플리케이션에 적용하기 전 검토

---
## 5. 파일 전송과 라이프사이클

### 파일 접근 방법

| 방법 | 설명 |
|------|------|
| 에이전트 파일시스템 도구 | `execute()`를 통한 직접 파일 작업 |
| 파일 전송 API | `uploadFiles()`, `downloadFiles()`로 시드/아티팩트 관리 |

### 라이프사이클 관리
샌드박스는 불필요한 비용을 방지하기 위해 **명시적 종료**가 필요합니다.  
채팅 애플리케이션에서는 대화 스레드별 고유 샌드박스에 TTL(Time-to-Live) 설정을 사용합니다.

In [ ]:
# 파일 전송과 라이프사이클 설정 예시 (참고용)
lifecycle_config = {
    "ttl_seconds": 1800,  # 30분
    "auto_shutdown": True,
    "thread_isolation": True,
}

file_operations = [
    "uploadFiles(['/local/data.csv'], '/sandbox/data/')",
    "downloadFiles(['/sandbox/output/result.json'], '/local/results/')",
]

print("=== 라이프사이클 설정 ===")
for key, value in lifecycle_config.items():
    print(f"  {key}: {value}")

print("\n=== 파일 전송 예시 ===")
for op in file_operations:
    print(f"  {op}")

---
## 6. ACP 개요

**ACP(Agent Client Protocol)**는 코딩 에이전트와 개발 환경(에디터/IDE) 간의 통신을 표준화하는 프로토콜입니다.

### MCP vs ACP

| 프로토콜 | 용도 | 대상 |
|---------|------|------|
| **MCP** (Model Context Protocol) | 외부 도구 통합 | 에이전트 ↔ 외부 서비스 |
| **ACP** (Agent Client Protocol) | 에디터-에이전트 통합 | 에이전트 ↔ 에디터/IDE |

ACP는 에이전트가 에디터와 직접 상호작용하여 코드 편집, 파일 탐색, 터미널 명령을 수행할 수 있게 합니다.

---
## 7. ACP 서버 구현

In [ ]:
# ACP 서버 구현 예시 (참고용)
acp_server_code = """
# pip install deepagents-acp
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

# 에이전트 생성
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    checkpointer=MemorySaver(),
)

# ACP 서버 실행 (stdio 모드)
server = AgentServerACP(agent)
server.run()
"""

print("=== ACP 서버 구현 예시 ===")
print(acp_server_code)

print("설치: pip install deepagents-acp")
print("실행: python acp_server.py (stdio 모드)")

---
## 8. ACP 지원 에디터

| 에디터 | 통합 방식 |
|--------|----------|
| **Zed** | 네이티브 통합 |
| **JetBrains IDEs** | 빌트인 지원 |
| **Visual Studio Code** | vscode-acp 플러그인 |
| **Neovim** | ACP 호환 플러그인 |

### Zed 설정 예시

```json
// Zed settings.json
{
  "agent_servers": [
    {
      "command": "python",
      "args": ["acp_server.py"],
      "env": {
        "ANTHROPIC_API_KEY": "sk-..."
      }
    }
  ]
}
```

### 추가 도구: Toad
**Toad**는 ACP 서버를 로컬 개발 도구로 실행하기 위한 프로세스 관리자입니다.  
`uv`를 통해 설치할 수 있습니다.

---
## 9. 샌드박스 + ACP 통합

샌드박스와 ACP를 결합하면 에디터에서 에이전트를 제어하면서, 코드 실행은 격리된 환경에서 수행하는 **완전한 아키텍처**를 구현할 수 있습니다.

### 통합 아키텍처

```
[에디터/IDE] <-- ACP --> [에이전트] <-- API --> [샌드박스]
    |                       |                      |
  코드 편집              태스크 관리            코드 실행
  파일 탐색              컨텍스트 관리          파일 격리
  터미널 UI              도구 호출              보안 환경
```

### 장점
- 에디터에서 직접 에이전트와 상호작용
- 코드 실행은 안전한 샌드박스에서 수행
- 시크릿은 호스트(에이전트 측)에서만 관리

In [ ]:
# 샌드박스 + ACP 통합 예시 (참고용)
integrated_config = """
from deepagents import create_deep_agent
from deepagents.backends.sandbox import ModalSandbox
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

# 샌드박스 백엔드 + ACP 서버 통합
agent = create_deep_agent(
    model="gpt-4.1",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    backend=ModalSandbox(image="python:3.12-slim"),
    checkpointer=MemorySaver(),
    interrupt_on={"execute": True},  # 코드 실행 전 승인
)

# ACP로 에디터와 연결
server = AgentServerACP(agent)
server.run()
"""

print("=== 샌드박스 + ACP 통합 예시 ===")
print(integrated_config)

print("이 구성의 효과:")
print("  1. 에디터에서 ACP를 통해 에이전트와 상호작용")
print("  2. 코드 실행은 Modal 샌드박스에서 안전하게 수행")
print("  3. execute 호출 시 Human-in-the-Loop 승인 필요")

---
## 전체 요약

| 주제 | 핵심 개념 | 핵심 API/도구 |
|------|----------|-------------|
| 샌드박스 개념 | 격리된 실행 환경으로 호스트 보호 | `execute`, 파일시스템 도구 |
| 아키텍처 패턴 | Agent-in-Sandbox vs Sandbox-as-Tool | Sandbox-as-Tool 권장 |
| 프로바이더 | Modal(GPU), Daytona(빠른 시작), Runloop(일회용) | `ModalSandbox` |
| 보안 | 시크릿 외부 관리, HITL, 네트워크 차단 | `interrupt_on` |
| ACP 개요 | 에디터-에이전트 통신 표준화 | `AgentServerACP` |
| ACP 서버 | stdio 모드로 에이전트 노출 | `deepagents-acp` |
| 에디터 통합 | Zed, JetBrains, VS Code, Neovim | ACP 프로토콜 |
| 통합 패턴 | 에디터 ↔ 에이전트 ↔ 샌드박스 | ACP + Sandbox 결합 |

### 다음 단계
→ **[고급 과정](../05_advanced/00_migration.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [Sandboxes](../docs/deepagents/11-sandboxes.md)
- [Agent Client Protocol (ACP)](../docs/deepagents/14-acp.md)